# Phase 3C: Route-Level Service & Accessibility Analytics

This notebook analyzes scheduled GTFS service supply. It does not measure passenger accessibility, demand, ridership, equity, usage, or actual operations.

## Data availability and limitations

The processed GTFS Parquet tables are absent from this repository clone. When available locally, the notebook reads them without modifying them. Otherwise it uses a clearly labelled synthetic fixture. Synthetic outputs are method examples, not Hyderabad findings. Date-based metrics require calendar bounds and complete weekday flags; explicit `calendar_dates` exceptions are applied when supplied. The fixture omits date bounds, so those metrics remain undefined.

In [ ]:
from pathlib import Path
import pandas as pd
from src.analysis.gtfs_service import build_service_accessibility_analytics

processed_dir = Path('data') / 'processed' / 'gtfs'
paths = sorted(processed_dir.glob('*.parquet'))
if paths:
    tables = {path.stem: pd.read_parquet(path) for path in paths}
    input_description = 'Existing local processed GTFS tables'
else:
    input_description = 'Synthetic illustrative fixture (not real-feed data)'
    tables = {
        'routes': pd.DataFrame({'feed_id': ['SYN','SYN'], 'route_id': ['SYN:r1','SYN:r2']}),
        'stops': pd.DataFrame({'feed_id': ['SYN']*3, 'stop_id': ['SYN:s1','SYN:s2','SYN:s3']}),
        'trips': pd.DataFrame({'feed_id': ['SYN']*3, 'trip_id': ['SYN:t1','SYN:t2','SYN:t3'], 'route_id': ['SYN:r1','SYN:r1','SYN:r2'], 'service_id': ['SYN:w']*3}),
        'stop_times': pd.DataFrame({'feed_id': ['SYN']*4, 'trip_id': ['SYN:t1','SYN:t1','SYN:t2','SYN:t3'], 'stop_id': ['SYN:s1','SYN:s2','SYN:s1','SYN:s2'], 'stop_sequence': [1,2,1,1], 'departure_time': ['07:00:00','07:20:00','08:00:00','25:30:00']}),
        'route_stops': pd.DataFrame({'feed_id': ['SYN']*3, 'route_id': ['SYN:r1','SYN:r1','SYN:r2'], 'stop_id': ['SYN:s1','SYN:s2','SYN:s2'], 'trip_count': [2,2,1], 'service_count': [1,1,1]}),
        'route_service_summary': pd.DataFrame({'feed_id': ['SYN']*2, 'route_id': ['SYN:r1','SYN:r2'], 'first_departure_seconds': [25200,91800], 'last_arrival_seconds': [30000,93000]}),
        'stop_service_summary': pd.DataFrame({'feed_id': ['SYN']*3, 'stop_id': ['SYN:s1','SYN:s2','SYN:s3'], 'route_count': [1,2,0], 'trip_count': [2,3,0]}),
        'calendar': pd.DataFrame({'feed_id': ['SYN'], 'service_id': ['SYN:w'], 'monday': [1], 'tuesday': [1], 'wednesday': [1], 'thursday': [1], 'friday': [1], 'saturday': [0], 'sunday': [0]}),
    }
print(input_description)

In [ ]:
analytics = build_service_accessibility_analytics(tables)
analytics['route_service_profile'].head()

In [ ]:
analytics['stop_service_profile'].head()

In [ ]:
analytics['temporal_service_profile'].head(12)

In [ ]:
analytics['feed_service_summary']  # Descriptive summaries, no feed ranking.

In [ ]:
analytics['quality_report']

## Interpretation boundary

The configurable score summarizes schedule intensity, route overlap, scheduled span, and stop count only. Gap flags use analyst-selected thresholds. Neither is a measure of passenger accessibility or demand. See `docs/SERVICE_ACCESSIBILITY_ANALYSIS.md` for formulas, missing-data behavior, calendar semantics, and limitations.